# Atelier Matplotlib – Visualisation de données de capteurs IoT

Après avoir manipulé les données avec NumPy et Pandas, on les **représente
graphiquement** pour identifier des tendances, comparer les bâtiments et
repérer des anomalies.

## Mise en place
- **Objectif** : importer les bibliothèques et charger le dataset.
- `matplotlib.pyplot` est le module de tracé, importé sous le surnom `plt`.
- On convertit `date_heure` en vrai type date, indispensable pour les courbes
  dans le temps, puis on trie les mesures par ordre chronologique.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/mesures_capteurs.csv")

# Conversion de la colonne date en type datetime, puis tri chronologique
df["date_heure"] = pd.to_datetime(df["date_heure"])
df = df.sort_values("date_heure")

df.head()

## Partie 1 – Graphique linéaire (Line Plot)

Une courbe relie les points dans l'ordre : idéale pour suivre l'évolution d'une
grandeur **dans le temps**.

### 1. Évolution de la température en fonction du temps
- **Objectif** : tracer une courbe temps → température, avec titre, labels,
  légende et grille.
- **Syntaxe** :
  - `plt.plot(x, y, label=...)` → trace la courbe
  - `plt.title(...)`, `plt.xlabel(...)`, `plt.ylabel(...)` → titre et labels
  - `plt.legend()` → affiche la légende, `plt.grid(True)` → affiche la grille

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(df["date_heure"], df["temperature"], color="tab:red", label="Température (°C)")
plt.title("Évolution de la température en fonction du temps")
plt.xlabel("Date et heure")
plt.ylabel("Température (°C)")
plt.legend()
plt.grid(True)
plt.show()

### 2. Identifier visuellement les valeurs anormales

Sur la courbe, on repère facilement des **pics isolés** : une température proche
de **58 °C** et une autre proche de **−18 °C**. Physiquement improbables pour un
bâtiment, ce sont des **mesures anormales** (capteurs défaillants ou erreurs de
relevé) — exactement le type de situations que le futur système de ML devra détecter.

## Partie 2 – Diagramme en barres (Bar Chart)

Un diagramme en barres compare des **catégories** entre elles : ici, la
consommation moyenne de chaque bâtiment.

### 1 & 2. Consommation moyenne par bâtiment (barres verticales)
- **Objectif** : calculer la moyenne par bâtiment, puis la représenter.
- **Syntaxe** : `plt.bar(categories, valeurs)`

In [ ]:
consommation_batiment = df.groupby("batiment")["consommation"].mean()

plt.figure(figsize=(8, 5))
plt.bar(consommation_batiment.index, consommation_batiment.values, color="tab:blue")
plt.title("Consommation moyenne par bâtiment")
plt.xlabel("Bâtiment")
plt.ylabel("Consommation moyenne")
plt.grid(axis="y")
plt.show()

### 3. Version horizontale
- **Objectif** : mêmes données, barres horizontales.
- **Syntaxe** : `plt.barh(categories, valeurs)`

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(consommation_batiment.index, consommation_batiment.values, color="tab:green")
plt.title("Consommation moyenne par bâtiment")
plt.xlabel("Consommation moyenne")
plt.ylabel("Bâtiment")
plt.grid(axis="x")
plt.show()

### 4. Quand le graphique horizontal est-il plus lisible ?

Le graphique **horizontal** est préférable quand :
- les **noms de catégories sont longs** (ils se lisent à l'horizontale sans se chevaucher) ;
- il y a **beaucoup de catégories** (l'empilement vertical reste lisible) ;
- on veut mettre en avant un **classement** (du plus grand au plus petit).

## Partie 3 – Histogramme

Un histogramme montre la **distribution** d'une variable : on découpe la plage
de valeurs en intervalles (les *classes* ou *bins*) et on compte combien de
valeurs tombent dans chacun.

### 1. Distribution des températures (20 classes)
- **Objectif** : visualiser la répartition des températures.
- **Syntaxe** : `plt.hist(donnees, bins=nombre_de_classes)`
- On retire d'abord les valeurs manquantes avec `.dropna()`.
- `plt.grid(axis="y")` n'affiche que les **lignes horizontales** de la grille.

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(df["temperature"].dropna(), bins=20, color="tab:orange", edgecolor="black")
plt.title("Distribution des températures")
plt.xlabel("Température (°C)")
plt.ylabel("Nombre de mesures")
plt.grid(axis="y")
plt.show()

### 2. Analyse de la distribution des températures

- **Concentration** : la grande majorité des températures se situe autour de
  **23–26 °C**.
- **Symétrie** : la distribution est globalement **symétrique** (forme en cloche).
- **Valeurs éloignées** : on distingue quelques valeurs très à l'écart, loin du cœur.
- **Valeurs aberrantes** : oui, les extrêmes (~−18 °C et ~58 °C) sont clairement
  aberrants.

### 3. Distribution de la consommation (20, puis 10, puis 30 classes)
- **Objectif** : observer l'effet du nombre de classes sur la lecture.

In [ ]:
for n in [20, 10, 30]:
    plt.figure(figsize=(9, 4))
    plt.hist(df["consommation"].dropna(), bins=n, color="tab:purple", edgecolor="black")
    plt.title(f"Distribution de la consommation ({n} classes)")
    plt.xlabel("Consommation")
    plt.ylabel("Nombre de mesures")
    plt.grid(axis="y")
    plt.show()

### c. Impact du nombre de classes

- **Peu de classes (10)** : histogramme lisse, vue d'ensemble, mais on perd des détails.
- **Beaucoup de classes (30)** : plus de détails, mais l'histogramme devient
  « bruité » et plus difficile à interpréter.
- Le nombre de classes est donc un **compromis** entre lisibilité globale et finesse.

## Partie 4 – Nuage de points (Scatter Plot)

Un nuage de points affiche chaque mesure comme un point (x, y). Il sert à
repérer une éventuelle **relation entre deux variables**.

### 1 & 2. Température vs consommation
- **Objectif** : chercher une relation entre température et consommation.
- **Syntaxe** : `plt.scatter(x, y)`
- `alpha=0.5` rend les points semi-transparents (utile quand ils se superposent).

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(df["temperature"], df["consommation"], alpha=0.5, color="tab:blue")
plt.title("Température vs Consommation")
plt.xlabel("Température (°C)")
plt.ylabel("Consommation")
plt.grid(True)
plt.show()

**Interprétation** : les points sont assez **dispersés**, sans tendance nette
(ni droite montante ni descendante). Il ne semble donc **pas exister de relation
forte** entre la température et la consommation dans ce jeu de données.